# 🏏 India vs Pakistan CT 2025 — Twitter Analysis & Business Insights

> **Full pipeline:** Data cleaning → Bot detection → Engagement analysis → Sentiment (VADER) → Topic modeling (BERTopic) → Business insights

## ⚙️ Setup

Set your Gemini API key as an environment variable before running:

```bash
export GEMINI_API_KEY='your-key-here'
```

Or in Colab, use the Secrets panel (🔑) to store `GEMINI_API_KEY`.

---

**Dataset:** `indvspak_master.csv` (not included — see README for structure)  
**Author:** [Your Name]  
**Date:** February 2026


In [ ]:
# Install dependencies
!pip install pandas numpy tqdm langdetect deep-translator
!pip install vaderSentiment transformers torch
!pip install bertopic matplotlib seaborn plotly kaleido wordcloud
!pip install google-generativeai
!pip install vaderSentiment wordcloud plotly kaleido --quiet

# ⚠️  API KEY — DO NOT hardcode keys here.
# Use environment variable or Colab Secrets:
# import os
# GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
# Or in Colab: from google.colab import userdata; GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')


In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import re
import warnings
import unicodedata
from collections import Counter

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from wordcloud import WordCloud
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

warnings.filterwarnings('ignore')

# ── Global Style ──────────────────────────────────────────────────────────────
PLOTLY_THEME = 'plotly_white'
FONT         = dict(family='Arial', size=13)
COLOR_POS    = '#2ECC71'
COLOR_NEG    = '#E74C3C'
COLOR_NEU    = '#95A5A6'
COLOR_MAIN   = '#1A56DB'
COLOR_ACC    = '#F39C12'
PALETTE      = px.colors.qualitative.Bold

def save(fig, name):
    fig.update_layout(template=PLOTLY_THEME, font=FONT)
    fig.show()

print('✅ Imports successful')

In [ ]:
df = pd.read_csv('indvspak_master.csv')
df['UTC_Time'] = pd.to_datetime(df['UTC_Time'], utc=True)
df['IST_Time'] = df['UTC_Time'] + pd.Timedelta(hours=5, minutes=30)
df['hour_utc'] = df['UTC_Time'].dt.hour
df['hour_ist'] = df['IST_Time'].dt.hour
df['date']     = df['UTC_Time'].dt.date

# Working datasets
organic = df[df['is_organic']].copy()
bots    = df[df['likely_bot']].copy()

# Derived columns
organic['engagement_total'] = organic['Like_Count'] + organic['Repost_Count'] + organic['Reply_Count']
organic['eng_rate']         = np.where(organic['View_Count'] > 0,
                                  (organic['engagement_total'] / organic['View_Count']) * 100, 0)
organic['hashtag_count']    = organic['Tweet_Content'].str.count('#')
organic['tweet_len']        = organic['Tweet_Content'].str.len()

print(f'✅ Loaded: {len(df):,} total tweets')
print(f'   Organic:    {len(organic):,}')
print(f'   Bot/spam:   {len(bots):,}')
print(f'   Date range: {df["UTC_Time"].min().date()} → {df["UTC_Time"].max().date()}')
df.head(3)

In [ ]:
# ── Audit Numbers ─────────────────────────────────────────────────────────────
raw_count     = 12508
unique_count  = len(df)
dup_count     = raw_count - unique_count
bot_count     = df['likely_bot'].sum()
organic_count = df['is_organic'].sum()

print('=' * 52)
print('  DATA QUALITY AUDIT')
print('=' * 52)
print(f'  Raw rows collected:        {raw_count:,}')
print(f'  Cross-file duplicates:     {dup_count:,}  ({dup_count/raw_count*100:.1f}%)')
print(f'  Unique tweets:             {unique_count:,}')
print(f'  Bot / inauthentic:         {bot_count:,}  ({bot_count/unique_count*100:.1f}%)')
print(f'  ✅ Organic (clean):        {organic_count:,}  ({organic_count/unique_count*100:.1f}%)')
print('=' * 52)

signals = {
    'Zero Engagement':    int(df['zero_engagement'].sum()),
    'Only Hashtags':      int(df['only_hashtags'].sum()),
    'Suspicious Language':int(df['suspicious_lang'].sum()),
    'Coordinated Text':   int(df['coordinated_text'].sum()),
    'Flagged (2+ signals)':int(df['likely_bot'].sum()),
}

fig = go.Figure(go.Bar(
    x=list(signals.values()),
    y=list(signals.keys()),
    orientation='h',
    marker_color=['#FF6B6B','#FF6B6B','#FF6B6B','#FF6B6B','#C0392B'],
    text=[f'{v:,}' for v in signals.values()],
    textposition='outside'
))
fig.update_layout(
    title='🤖 Inauthentic Signal Breakdown — What Was Hiding in the Raw Data',
    xaxis_title='Tweet Count', height=380
)
save(fig, 'audit_signals')

In [ ]:
# ── Bot Signal Score Distribution ─────────────────────────────────────────────
score_dist = df['bot_signals'].value_counts().sort_index().reset_index()
score_dist.columns = ['signals_triggered', 'tweet_count']

fig = px.bar(
    score_dist, x='signals_triggered', y='tweet_count',
    color='signals_triggered',
    color_continuous_scale=['#2ECC71','#F39C12','#E67E22','#E74C3C','#8E44AD'],
    title='Bot Signal Score Distribution — 0 = Clean, 4 = Maximum Suspicion',
    labels={'signals_triggered': 'Number of Bot Signals Triggered', 'tweet_count': 'Tweet Count'},
    text='tweet_count'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=400)
save(fig, 'bot_score_dist')

print('\n💡 BUSINESS INSIGHT:')
print('  4,077 tweets had exactly 1 bot signal — too many to be bots, but worth noting.')
print('  Only tweets with 2+ signals were removed. Conservative threshold = cleaner data.')
print(f'  Result: {organic_count:,} verified organic tweets vs the raw inflated {raw_count:,}.')

In [ ]:
# ── Funnel Chart: Raw → Clean ──────────────────────────────────────────────────
stages  = ['Raw Scraped', 'After Dedup', 'After Bot Removal (Organic)']
counts  = [raw_count, unique_count, organic_count]
colors  = ['#E74C3C', '#F39C12', '#2ECC71']

fig = go.Figure(go.Funnel(
    y=stages, x=counts,
    textinfo='value+percent initial',
    marker_color=colors
))
fig.update_layout(
    title='Data Cleaning Funnel — From Raw to Trustworthy',
    height=380
)
save(fig, 'cleaning_funnel')

# ENGAGEMENT ANALYSIS


In [ ]:
# ── Engagement Distribution (Log Scale) ───────────────────────────────────────
fig = make_subplots(rows=1, cols=3,
    subplot_titles=('Like Count Distribution', 'Repost Count Distribution', 'View Count Distribution'))

for col, metric in enumerate(['Like_Count','Repost_Count','View_Count'], 1):
    data = organic[organic[metric]>0][metric]
    fig.add_trace(go.Histogram(
        x=np.log10(data+1),
        marker_color=COLOR_MAIN,
        opacity=0.8,
        name=metric
    ), row=1, col=col)

fig.update_layout(
    title='Engagement Distribution (Log10 Scale) — Extreme Power Law in All Metrics',
    showlegend=False, height=400
)
save(fig, 'engagement_dist')

print('\n📊 ENGAGEMENT SUMMARY STATISTICS:')
stats = organic[['Like_Count','Repost_Count','View_Count','Reply_Count','Bookmark_Count']].describe().round(1)
print(stats)

In [ ]:
# ── Engagement Tier Breakdown ──────────────────────────────────────────────────
organic['eng_tier'] = pd.cut(
    organic['engagement_total'],
    bins=[-1, 0, 5, 50, 500, 999999],
    labels=['Zero (0)', 'Low (1–5)', 'Medium (6–50)', 'High (51–500)', 'Viral (500+)']
)

tier_counts = organic['eng_tier'].value_counts().sort_index().reset_index()
tier_counts.columns = ['tier', 'count']
tier_counts['pct'] = (tier_counts['count'] / len(organic) * 100).round(1)

fig = px.bar(
    tier_counts, x='tier', y='count',
    color='tier',
    color_discrete_sequence=['#BDC3C7','#85C1E9','#F39C12','#E74C3C','#8E44AD'],
    text=tier_counts.apply(lambda r: f"{r['count']:,}\n({r['pct']}%)", axis=1),
    title='Engagement Tier Breakdown — Most Tweets Are Invisible'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=430, xaxis_title='Engagement Tier', yaxis_title='Tweet Count')
save(fig, 'eng_tiers')

💡 BUSINESS INSIGHT:
  43% of organic tweets got ZERO engagement (likes + RTs + replies = 0).
  Only 1.3% of tweets went viral (500+ combined engagement).
  
  
  

>***This means: being on the right platform at the right time matters more than volume.***





##  2. Power Law & Influence


In [ ]:
# ── Lorenz Curve (Inequality of Engagement) ────────────────────────────────────
author_likes = organic.groupby('Author_Handle')['Like_Count'].sum().sort_values()
n = len(author_likes)
cumulative_likes   = np.cumsum(author_likes.values) / author_likes.sum()
cumulative_authors = np.arange(1, n+1) / n

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=cumulative_authors*100, y=cumulative_likes*100,
    name='Actual Distribution', line=dict(color=COLOR_MAIN, width=3)
))
fig.add_trace(go.Scatter(
    x=[0,100], y=[0,100],
    name='Perfect Equality', line=dict(color='gray', dash='dash')
))
fig.add_vline(x=99, line_dash='dot', line_color=COLOR_NEG, annotation_text='Top 1% = 85.4% of likes')
fig.update_layout(
    title='Lorenz Curve — Extreme Inequality of Engagement in Cricket Twitter',
    xaxis_title='% of Accounts (cumulative)',
    yaxis_title='% of Total Likes (cumulative)',
    height=450
)
save(fig, 'lorenz_curve')

# Print power law numbers
total_likes = author_likes.sum()
for pct in [1, 5, 10]:
    n_authors = int(n * pct / 100)
    share = author_likes.iloc[-n_authors:].sum() / total_likes * 100
    print(f'Top {pct:2d}% ({n_authors:4d} accounts) → {share:.1f}% of all likes')

Business Insight:

- In cricket Twitter, influence is not distributed — it's concentrated.

- The top 1% of accounts (52 accounts) drove 85.4% of ALL likes.

- A brand targeting this conversation only needs to reach ~50 accounts to dominate the narrative.

In [ ]:
# ── Top 15 Influencers ────────────────────────────────────────────────────────
top_authors = organic.groupby('Author_Handle').agg(
    tweets       = ('Post_ID',     'count'),
    total_likes  = ('Like_Count',  'sum'),
    total_views  = ('View_Count',  'sum'),
    total_rts    = ('Repost_Count','sum'),
    verified     = ('Verified_Status', 'first')
).reset_index().sort_values('total_likes', ascending=False).head(15)

top_authors['verified_label'] = top_authors['verified'].map({True:'✅ Verified', False:'❌ Unverified'})

fig = px.bar(
    top_authors, x='total_likes', y='Author_Handle',
    orientation='h',
    color='verified_label',
    color_discrete_map={'✅ Verified': COLOR_MAIN, '❌ Unverified': COLOR_ACC},
    hover_data=['tweets','total_views','total_rts'],
    title='Top 15 Influencers by Total Likes — All Top Accounts Are Verified',
    text='total_likes'
)
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, height=500, xaxis_title='Total Likes')
save(fig, 'top_influencers')

💡 BUSINESS INSIGHT:
  
  - RealWahidaAFG (Afghan account) is the #1 most liked account — not an Indian or Pakistani handle.
  - The Afghanistan narrative was the single biggest organic engagement driver in the dataset.
  
  *A brand could have tapped into this cross-border goodwill and gone massively viral.*

##  3. Verified vs Unverified Accounts

In [ ]:
# ── Verified vs Unverified — All Metrics ──────────────────────────────────────
ver_stats = organic.groupby('Verified_Status').agg(
    tweet_count  = ('Post_ID',       'count'),
    avg_likes    = ('Like_Count',    'mean'),
    avg_rts      = ('Repost_Count',  'mean'),
    avg_views    = ('View_Count',    'mean'),
    avg_replies  = ('Reply_Count',   'mean'),
    avg_eng_rate = ('eng_rate',      'mean')
).round(2).reset_index()
ver_stats['label'] = ver_stats['Verified_Status'].map({True:'✅ Verified', False:'❌ Unverified'})

metrics = ['avg_likes', 'avg_rts', 'avg_views', 'avg_replies']
titles  = ['Avg Likes', 'Avg Retweets', 'Avg Views', 'Avg Replies']

fig = make_subplots(rows=2, cols=2, subplot_titles=titles)
positions = [(1,1),(1,2),(2,1),(2,2)]

for (row,col), metric, title in zip(positions, metrics, titles):
    fig.add_trace(go.Bar(
        x=ver_stats['label'],
        y=ver_stats[metric],
        marker_color=[COLOR_MAIN, COLOR_NEG],
        text=ver_stats[metric].round(1),
        textposition='outside',
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    title='✅ Verified vs ❌ Unverified — Engagement Across All Metrics (Organic Tweets)',
    height=550
)
save(fig, 'verified_engagement')

v = ver_stats[ver_stats['Verified_Status']==True].iloc[0]
u = ver_stats[ver_stats['Verified_Status']==False].iloc[0]

💡 BUSINESS INSIGHT:
  
 -  Verified accounts get 23x more likes per tweet
 -  Verified accounts get 6x more views per tweet
  Engagement Rate:
  
   *Verified 1.59% vs Unverified 0.71%
  Despite being only 40% of accounts, verified users dominate total reach.*

##  4. Media vs Text-Only Tweets

 Tweets with images or videos get 3.8x more likes and 3.3x more views than text-only tweets. For any sports campaign — visual content is non-negotiable.

In [ ]:
# ── Media vs Text — 2x2 by Verified Status ────────────────────────────────────
media_stats = organic.groupby(['has_media','Verified_Status']).agg(
    count       = ('Post_ID',       'count'),
    avg_likes   = ('Like_Count',    'mean'),
    avg_views   = ('View_Count',    'mean'),
    avg_rts     = ('Repost_Count',  'mean')
).round(1).reset_index()

media_stats['group'] = media_stats.apply(
    lambda r: f"{'Media' if r['has_media'] else 'Text'} + {'Verified' if r['Verified_Status'] else 'Unverified'}",
    axis=1
)

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('Avg Likes', 'Avg Views', 'Avg Retweets'))
colors = ['#E74C3C','#F39C12','#85C1E9','#1A56DB']

for col, metric in enumerate(['avg_likes','avg_views','avg_rts'], 1):
    fig.add_trace(go.Bar(
        x=media_stats['group'],
        y=media_stats[metric],
        marker_color=colors,
        text=media_stats[metric],
        textposition='outside',
        showlegend=False
    ), row=1, col=col)

fig.update_layout(
    title='🎬 Media vs Text × Verified vs Unverified — Engagement Comparison',
    height=440
)
fig.update_xaxes(tickangle=25)
save(fig, 'media_vs_text')

text_likes  = organic[organic['has_media']==False]['Like_Count'].mean()
media_likes = organic[organic['has_media']==True]['Like_Count'].mean()

In [ ]:
print(f'\n💡 BUSINESS INSIGHT:')
print(f'  Media tweets: {media_likes:.1f} avg likes vs Text tweets: {text_likes:.1f} avg likes')
print(f'  Media multiplier: {media_likes/text_likes:.1f}x more likes')
print(f'  Verified + Media = the ultimate combo: 120.9 avg likes vs 2.0 for Unverified + Text')
print(f'  Brands without visual content in live sports events are invisible.')

💡 BUSINESS INSIGHT:
  - Media tweets: 57.4 avg likes vs Text tweets: 14.9 avg likes
  - Media multiplier: 3.9x more likes
  - Verified + Media = the ultimate combo: 120.9 avg likes vs 2.0 for Unverified + Text

>  *Brands without visual content in live sports events are invisible.*




##  5. Hashtag & Trend Analysis

Secondary hashtags reveal what conversations are being ATTACHED to cricket. #Daytona500 and #RISERConcert appearing 137+ and 300+ times shows how sports events get hijacked for unrelated traffic.

In [ ]:
# ── Extract & Rank Hashtags ────────────────────────────────────────────────────
def extract_hashtags(text):
    if not isinstance(text, str): return []
    return [h.lower() for h in re.findall(r'#(\w+)', text)]

organic['hashtag_list'] = organic['Tweet_Content'].apply(extract_hashtags)
all_tags = [t for tags in organic['hashtag_list'] for t in tags]
top_tags = pd.DataFrame(Counter(all_tags).most_common(30), columns=['hashtag','count'])

# Remove primary search hashtags for secondary analysis
primary = ['indvspak','indvpak','t20worldcup','t20worldcup2026','indiavspakistan']
secondary = top_tags[~top_tags['hashtag'].isin(primary)].head(20)

fig = px.bar(
    secondary.sort_values('count'),
    x='count', y='hashtag', orientation='h',
    color='count',
    color_continuous_scale='Blues',
    title='Top 20 Co-occurring Hashtags — Excluding Primary Search Tags',
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=580, xaxis_title='Occurrences')
save(fig, 'top_hashtags')


💡 INSIGHT:
  - #IshanKishan (713) and #SuryakumarYadav (629) → India batting dominated the narrative
  - #BabarAzam (159) → mostly negative sentiment — Pakistan player bore the blame
  - #RISERConcert (300+) and #Daytona500 (137) → off-topic hashtag injection for reach
  

In [ ]:
# ── Hashtag Count vs Engagement ────────────────────────────────────────────────
bins   = pd.cut(organic['hashtag_count'], bins=[-1,1,3,5,100], labels=['1','2–3','4–5','6+'])
ht_eng = organic.groupby(bins).agg(
    tweet_count  = ('Post_ID',       'count'),
    avg_likes    = ('Like_Count',    'mean'),
    avg_views    = ('View_Count',    'mean'),
    avg_rts      = ('Repost_Count',  'mean')
).round(1).reset_index()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=('Avg Likes by Hashtag Count', 'Avg Views by Hashtag Count'))
for col, metric in enumerate(['avg_likes','avg_views'], 1):
    fig.add_trace(go.Bar(
        x=ht_eng['hashtag_count'].astype(str),
        y=ht_eng[metric],
        marker_color=[COLOR_MAIN, COLOR_ACC, COLOR_NEG, '#8E44AD'],
        text=ht_eng[metric], textposition='outside', showlegend=False
    ), row=1, col=col)

fig.update_layout(
    title='#️⃣ Hashtag Count vs Engagement — Fewer Hashtags = More Likes',
    height=400, xaxis_title='Hashtag Count', xaxis2_title='Hashtag Count'
)
save(fig, 'hashtag_count_engagement')


💡 BUSINESS INSIGHT:
  - Tweets with just 1 hashtag: 76.9 avg likes
  - Tweets with 6+ hashtags:    4.3 avg likes
  - Counter-intuitive finding: hashtag stuffing KILLS engagement.
  
  *Less is more. One focused hashtag > six spam hashtags.*

---
### 6. Language & Audience Geography

English dominates reach (60.7% of tweets, 76.8% of total likes). Hindi speakers (20.8%) are the second biggest audience — underserved by English-only brands. Urdu tweets have very low avg engagement (3.6 likes), suggesting Pakistani fans are either less active or algorithmically suppressed.

> ⚠️ **Tagalog Anomaly:** The raw data shows `tl` (Tagalog) tweets with 84.3 avg likes — but manual inspection reveals these are **not Filipino tweets**. Twitter's language detector mislabeled short, emoji-heavy, and Hinglish SMS-style tweets as Tagalog. The high average is driven by 2–3 viral Hindi tweets inflating a 203-tweet group (median likes = 1.0). **The raw `Language` column cannot be trusted — use the custom `final_lang` column for all language-based analysis.**

In [ ]:
# ── Language Distribution ─────────────────────────────────────────────────────
LANG_MAP = {
    'en':'English','hi':'Hindi','ur':'Urdu','tl':'Tagalog',
    'in':'Indonesian','ar':'Arabic','es':'Spanish','mr':'Marathi',
    'et':'Other','und':'Undefined','qme':'Media/URL Only','qht':'Hashtag Heavy'
}
organic['lang_label'] = organic['Language'].map(LANG_MAP).fillna('Other')

lang_stats = organic.groupby('lang_label').agg(
    tweets      = ('Post_ID',       'count'),
    avg_likes   = ('Like_Count',    'mean'),
    avg_views   = ('View_Count',    'mean'),
    total_likes = ('Like_Count',    'sum')
).round(1).reset_index().sort_values('tweets', ascending=False)

fig = make_subplots(rows=1, cols=2,
    specs=[[{'type':'domain'}, {'type':'xy'}]],
    subplot_titles=('Tweet Volume by Language', 'Avg Likes by Language'))

fig.add_trace(go.Pie(
    labels=lang_stats['lang_label'],
    values=lang_stats['tweets'],
    hole=0.4, textinfo='label+percent'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=lang_stats['lang_label'],
    y=lang_stats['avg_likes'],
    marker_color=PALETTE,
    text=lang_stats['avg_likes'].round(1),
    textposition='outside'
), row=1, col=2)

fig.update_layout(
    title='🌐 Language Breakdown — Volume vs Average Engagement',
    height=460, showlegend=False
)
save(fig, 'language_analysis')

# ── Tagalog Anomaly Investigation ───────────────────────────────────────────
tagalog = organic[organic["Language"] == "tl"]
print(f"\n⚠️  TAGALOG ANOMALY CHECK:")
print(f"  Total 'tl' tweets:  {len(tagalog)}")
print(f"  Avg likes:          {tagalog['Like_Count'].mean():.1f}")
print(f"  Median likes:       {tagalog['Like_Count'].median():.1f}  ← median tells the real story")


---
##  7. Match Timeline Analysis
**Business Insight:** 4,056 tweets (avg) were posted in a single hour (18:00–19:00 UTC = 23:30–00:30 IST). That is 67 tweets PER MINUTE during peak. Any brand not present in that window missed the entire conversation.

In [ ]:
# ── Hourly Volume + Avg Engagement ────────────────────────────────────────────
hourly = organic.groupby('hour_utc').agg(
    tweets    = ('Post_ID',       'count'),
    avg_likes = ('Like_Count',    'mean'),
    avg_views = ('View_Count',    'mean'),
    total_views=('View_Count',    'sum')
).reset_index().round(1)
hourly['hour_ist'] = hourly['hour_utc'].apply(lambda h: f"{(h+5)%24}:{30 if True else 0:02d} IST")

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
    subplot_titles=('Tweet Volume by Hour (UTC)', 'Avg Likes by Hour (UTC)'))

fig.add_trace(go.Bar(
    x=hourly['hour_utc'], y=hourly['tweets'],
    marker_color=COLOR_MAIN, name='Tweet Count',
    text=hourly['tweets'], textposition='outside'
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=hourly['hour_utc'], y=hourly['avg_likes'],
    mode='lines+markers', line=dict(color=COLOR_ACC, width=3),
    marker=dict(size=8), name='Avg Likes'
), row=2, col=1)

# Annotate peak
fig.add_vline(x=17, line_dash='dash', line_color=COLOR_NEG,
    annotation_text='17:00 UTC = Pre-match surge', annotation_position='top left', row=1, col=1)
fig.add_vline(x=18, line_dash='dash', line_color='purple',
    annotation_text='18:00 UTC = MATCH PEAK (4,056 tweets)', annotation_position='top right', row=1, col=1)

fig.update_layout(
    title='⏱️ Match Timeline — When Did Cricket Twitter Explode?',
    height=560, xaxis2_title='Hour (UTC)', showlegend=False
)
save(fig, 'timeline_hourly')

peak_hour = hourly.loc[hourly['tweets'].idxmax()]


💡 BUSINESS INSIGHT:
  - Peak hour: 18:00 UTC = 23:30 IST
  - Peak volume: 4,056 tweets in one hour (67 tweets/minute)
  - Pre-match (17:00 UTC) had highest avg likes: 64.9 — anticipation drives quality
  - After 20:00 UTC, volume drops 86% — live events are time-boxed opportunities

---
## 8. Tweet Length vs Performance

Short tweets (50–100 characters) get the most likes. Long tweets (300+) get more views but far fewer likes.

Format to your objective — short for engagement, long for reach.

In [ ]:
# ── Tweet Length Buckets ───────────────────────────────────────────────────────
len_bins   = pd.cut(organic['tweet_len'],
    bins=[0,50,100,150,200,300,999],
    labels=['<50','50–100','100–150','150–200','200–300','300+'])
len_stats = organic.groupby(len_bins).agg(
    count      = ('Post_ID',       'count'),
    avg_likes  = ('Like_Count',    'mean'),
    avg_views  = ('View_Count',    'mean'),
    avg_rts    = ('Repost_Count',  'mean')
).round(1).reset_index()

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('Avg Likes', 'Avg Views', 'Avg Retweets'))

for col, metric in enumerate(['avg_likes','avg_views','avg_rts'], 1):
    fig.add_trace(go.Bar(
        x=len_stats['tweet_len'].astype(str),
        y=len_stats[metric],
        marker_color=PALETTE,
        text=len_stats[metric], textposition='outside', showlegend=False
    ), row=1, col=col)

fig.update_layout(
    title='📝 Tweet Length vs Performance — Sweet Spot is 50–100 Characters',
    height=420, xaxis_title='Characters'
)
save(fig, 'tweet_length')


💡 BUSINESS INSIGHT:

  - 50–100 chars: 58.0 avg likes — the "punchline" length
  - <50 chars:    47.8 avg likes — short but punchy still works
  - 300+ chars:   18.1 avg likes — too long, people don't engage
  - 300+ chars:  1425 avg views — long tweets get seen but not liked
  
  *Lesson: If you want likes, be brief. If you want reach, write more.*

---
##  9. Sentiment Analysis (VADER)

 Negative tweets get 37% more likes and 21% more views than positive ones. Anger and criticism outperform celebration in terms of raw engagement — a sobering finding for brands trying to ride positive sports sentiment.

In [ ]:
# ── VADER Sentiment Scoring ────────────────────────────────────────────────────
sia = SentimentIntensityAnalyzer()

def vader_sentiment(text):
    if not isinstance(text, str): return 'neutral', 0.0
    score = sia.polarity_scores(text)['compound']
    if score >= 0.05:   return 'positive', score
    elif score <= -0.05: return 'negative', score
    else:               return 'neutral',   score

print('Running VADER on 9,552 organic tweets...')
results = organic['Tweet_Content'].apply(vader_sentiment)
organic['sentiment']       = results.apply(lambda x: x[0])
organic['sentiment_score'] = results.apply(lambda x: x[1])
print('✅ Sentiment analysis complete')
print(organic['sentiment'].value_counts())

In [ ]:
# ── Sentiment Distribution Chart ──────────────────────────────────────────────
sent_counts = organic['sentiment'].value_counts().reset_index()
sent_counts.columns = ['sentiment','count']
sent_counts['pct'] = (sent_counts['count']/len(organic)*100).round(1)
color_map = {'positive':COLOR_POS,'negative':COLOR_NEG,'neutral':COLOR_NEU}

fig = px.bar(
    sent_counts, x='sentiment', y='count',
    color='sentiment',
    color_discrete_map=color_map,
    text=sent_counts.apply(lambda r: f"{r['count']:,} ({r['pct']}%)", axis=1),
    title='💬 Overall Sentiment Distribution — 9,552 Organic Tweets'
)
fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=420, xaxis_title='Sentiment', yaxis_title='Tweet Count')
save(fig, 'sentiment_dist')

In [ ]:
# ── Sentiment vs Engagement ────────────────────────────────────────────────────
sent_eng = organic.groupby('sentiment').agg(
    count     = ('Post_ID',       'count'),
    avg_likes = ('Like_Count',    'mean'),
    avg_views = ('View_Count',    'mean'),
    avg_rts   = ('Repost_Count',  'mean')
).round(1).reset_index()

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('Avg Likes', 'Avg Views', 'Avg Retweets'))

for col, metric in enumerate(['avg_likes','avg_views','avg_rts'], 1):
    fig.add_trace(go.Bar(
        x=sent_eng['sentiment'],
        y=sent_eng[metric],
        marker_color=[color_map.get(s,'gray') for s in sent_eng['sentiment']],
        text=sent_eng[metric], textposition='outside', showlegend=False
    ), row=1, col=col)

fig.update_layout(
    title='🔥 Does Negative Content Outperform Positive? — Engagement by Sentiment',
    height=420
)
save(fig, 'sentiment_engagement')

neg = sent_eng[sent_eng['sentiment']=='negative'].iloc[0]
pos = sent_eng[sent_eng['sentiment']=='positive'].iloc[0]



💡 BUSINESS INSIGHT:
  - Negative tweets: 63.7 avg likes  vs  Positive tweets: 44.6 avg likes
  - Negative tweets: 1967 avg views vs  Positive tweets: 1097 avg views
  - Negativity bias is real — anger and criticism drive more engagement than celebration.
  
  *Brand strategy implication: reactive content (hot takes) > purely celebratory posts.*

In [ ]:
# ── Sentiment Over Time ────────────────────────────────────────────────────────
sent_hourly = organic.groupby(['hour_utc','sentiment']).size().reset_index(name='count')

fig = px.bar(
    sent_hourly, x='hour_utc', y='count', color='sentiment',
    color_discrete_map=color_map,
    title='💬 Sentiment Arc Over Time — How Did the Mood Shift Hour by Hour?',
    labels={'hour_utc':'Hour (UTC)','count':'Tweet Count'},
    barmode='stack'
)
fig.add_vline(x=18, line_dash='dash', line_color='black',
    annotation_text='Match Peak 18:00 UTC')
fig.update_layout(height=430)
save(fig, 'sentiment_timeline')

## 9. Virality Deep Dive — Top 10 Tweets


In [ ]:
# ── Top 10 Viral Tweets Table ──────────────────────────────────────────────────
top10 = organic.nlargest(10, 'Like_Count')[[
    'Tweet_Content','Like_Count','Repost_Count','View_Count',
    'Reply_Count','Verified_Status','Language','has_media'
]].reset_index(drop=True)

print('🏆 TOP 10 VIRAL TWEETS:\n')
for i, row in top10.iterrows():
    print(f"{'─'*70}")
    print(f"#{i+1} | ❤️ {row['Like_Count']:,} | 🔁 {row['Repost_Count']:,} | "
          f"👁 {row['View_Count']:,} | Verified: {row['Verified_Status']} | Lang: {row['Language']} | Media: {row['has_media']}")
    print(f"   {str(row['Tweet_Content'])[:150]}")

# Bubble Chart: Likes vs Views vs Retweets
top10['tweet_short'] = top10['Tweet_Content'].str[:40] + '...'

fig = px.scatter(
    top10,
    x='View_Count', y='Like_Count',
    size='Repost_Count',
    color='Language',
    hover_name='tweet_short',
    hover_data=['Repost_Count','Reply_Count'],
    title='🫧 Top 10 Viral Tweets — Views vs Likes (Bubble = Retweet Count)',
    size_max=60
)
fig.update_layout(height=480)
save(fig, 'viral_bubble')



💡 BUSINESS INSIGHT:
  - 3 of the top 5 tweets reference Afghanistan — not India or Pakistan
  - The #1 tweet (27,253 likes) was posted by a verified Afghan journalist
  - All top 10 tweets are from verified accounts — organic virality needs credibility
  - Highest view count (451K) was NOT the most liked — a controversial roast tweet

---
## 10. Coordinated & Bot Activity Deep Dive

In [ ]:
# ── Coordinated Text — What Were Bots Pushing? ────────────────────────────────
coord_tweets = df[df['coordinated_text']]['Tweet_Content'].value_counts().head(10).reset_index()
coord_tweets.columns = ['tweet','count']
coord_tweets['tweet_short'] = coord_tweets['tweet'].str[:60] + '...'

fig = px.bar(
    coord_tweets, x='count', y='tweet_short', orientation='h',
    color='count', color_continuous_scale='Reds',
    title='🤖 Top 10 Coordinated Texts — Same Tweet, Multiple Accounts',
    text='count'
)
fig.update_traces(textposition='outside')
fig.update_layout(
    yaxis={'categoryorder':'total ascending'},
    coloraxis_showscale=False, height=460
)
save(fig, 'coordinated_tweets')

# Bot vs Organic Engagement Comparison
compare = df.groupby('likely_bot').agg(
    count     = ('Post_ID',       'count'),
    avg_likes = ('Like_Count',    'mean'),
    avg_views = ('View_Count',    'mean'),
    avg_rts   = ('Repost_Count',  'mean')
).round(2).reset_index()
compare['label'] = compare['likely_bot'].map({True:'🤖 Bot', False:'✅ Organic'})

fig = make_subplots(rows=1, cols=3,
    subplot_titles=('Avg Likes','Avg Views','Avg Retweets'))
for col, metric in enumerate(['avg_likes','avg_views','avg_rts'], 1):
    fig.add_trace(go.Bar(
        x=compare['label'], y=compare[metric],
        marker_color=[COLOR_NEG, COLOR_POS],
        text=compare[metric], textposition='outside', showlegend=False
    ), row=1, col=col)
fig.update_layout(title='Bot vs Organic — Do Bots Even Drive Engagement?', height=400)
save(fig, 'bot_vs_organic')


💡 BUSINESS INSIGHT:
  - The #1 coordinated tweet: "#INDvsPAK" posted 172 times — pure hashtag spam
  - Bots had significantly lower avg likes AND views — they are noise, not signal
  - However, coordinated text CAN boost hashtag trend ranking — the mechanism is volume

  *Any analytics dashboard that doesn't filter bots is counting noise as insight*.

---
##  11. Correlation Matrix

 View count and like count are strongly correlated (0.72), but bot signals show near-zero correlation with engagement — confirming bots don't drive real interaction.

In [ ]:
# ── Correlation Matrix ────────────────────────────────────────────────────────
corr_cols = [
    'Like_Count','Repost_Count','View_Count','Reply_Count',
    'Bookmark_Count','hashtag_count','tweet_len','bot_signals'
]
corr = organic[corr_cols].corr().round(2)

fig = px.imshow(
    corr,
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='📐 Correlation Matrix — Engagement Metrics vs Tweet Features',
    text_auto=True
)
fig.update_layout(height=520)
save(fig, 'correlation_matrix')



💡 KEY CORRELATIONS:

  - Like ↔ View:         strong positive — visibility drives likes
  
  - Like ↔ Retweet:      strong positive — liked tweets get shared

  - Hashtag ↔ Like:      negative — more hashtags = fewer likes

  - bot_signals ↔ Like:  near zero — bots don't generate real engagement

  - tweet_len ↔ Like:    slight negative — longer tweets get fewer likes